# 04 Tabular models

**Owner:** Person B.

**Inputs:** `data/processed/features.parquet`, `outputs/column_roles.json`, `outputs/question_contract.json`.

**Outputs:** `splits/split.npz` (once), `data/processed/model_matrix.parquet`, `outputs/model_comparison.csv`, `outputs/models/selected_model.joblib`.

**Stop rule:** linear regression or logistic regression, then XGBoost, always run first if a model is needed. Set `RUN_LGBM` and `RUN_CATBOOST` to `True` only when the question is predictive and the clock is still before 14:05. The test split is never scored. If `model_needed` is false, stop without fitting.

The validation CSI is tuned on the validation scores. Quote PR-AUC or MAE as the selection metric, and do not call the validation number a test result.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import json

import pandas as pd

from src import io as sio
from src.features import aggregate_measures
from src.models import best_row, compare, design_matrix, detect_task, save_bundle
from src.roles import columns_with_role, read_roles
from src.split import load_split, save_split, split_exists, stratified_split, time_split

RUN_LGBM = False
RUN_CATBOOST = False

contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
if contract.get("model_needed") is False:
    print("STOP: the contract says no model is needed.")
elif not FEATURES_PATH.exists() and not FEATURES_PATH.with_suffix(".csv").exists():
    print("STOP: features.parquet is missing. Run notebook 03 first.")
else:
    frame = sio.read_frame(FEATURES_PATH).reset_index(drop=True)
    roles = read_roles()
    targets = [column for column in columns_with_role(roles, "target") if column in frame.columns]
    if not targets and contract.get("target") in frame.columns:
        targets = [contract["target"]]
    if not targets:
        print("STOP: tag a target column, or set model_needed to false.")
    else:
        target = targets[0]
        time_columns = [column for column in columns_with_role(roles, "time") if column in frame.columns]
        labelled = frame[target].notna().to_numpy()
        try:
            if split_exists():
                split = load_split()
                print("loaded frozen split", {part: int(len(idx)) for part, idx in split.items()})
            elif time_columns and contract.get("train_end") and contract.get("val_end"):
                split = time_split(frame[time_columns[0]], contract["train_end"], contract["val_end"], labelled=labelled)
                save_split(split, n=len(frame), note="time split from the question contract")
                print("wrote time split", {part: int(len(idx)) for part, idx in split.items()})
            else:
                split = stratified_split(frame[target])
                save_split(split, n=len(frame), note="stratified 70/15/15 because no time cutoff was set")
                print("wrote stratified split", {part: int(len(idx)) for part, idx in split.items()})
        except (AssertionError, ValueError, FileExistsError) as exc:
            split = None
            print("STOP:", type(exc).__name__, exc)
        if split is not None:
            measures = [column for column in columns_with_role(roles, "measure") if column in frame.columns and column != target]
            if "entity_from_descriptors" in frame.columns:
                entity_column = "entity_from_descriptors"
            else:
                ids = [column for column in columns_with_role(roles, "id") if column in frame.columns]
                entity_column = ids[0] if ids else None
            model_frame = frame
            if entity_column:
                aggregates = aggregate_measures(
                    frame,
                    entity_column,
                    measures,
                    time_column=time_columns[0] if time_columns else None,
                    train_positions=split["train"],
                )
                model_frame = pd.concat([frame.reset_index(drop=True), aggregates], axis=1)
            blocked = set(columns_with_role(roles, "id"))
            feature_columns = [
                column
                for column in model_frame.columns
                if column != target and column not in blocked and not str(column).startswith("entity_")
            ]
            built = design_matrix(model_frame, feature_columns, split["train"], split["val"])
            print("train matrix", built["X_train"].shape, "dropped", built["dropped"])
            families = ["linear", "xgboost"]
            if RUN_LGBM:
                families.append("lightgbm")
            if RUN_CATBOOST:
                families.append("catboost")
            y_train = model_frame.iloc[split["train"]][target]
            y_val = model_frame.iloc[split["val"]][target]
            task = detect_task(y_train)
            report, chosen = compare(
                built["X_train"], y_train, built["X_other"], y_val, families=tuple(families), task=task
            )
            report.to_csv(COMPARISON_PATH, index=False)
            print(report.to_string(index=False))
            print("Test rows were not scored.")
            winner = best_row(report)
            if winner is not None and winner["family"] in chosen:
                save_bundle(
                    BUNDLE_PATH,
                    chosen[winner["family"]],
                    built["preprocessor"],
                    built["feature_names"],
                    task,
                    target,
                    winner["family"],
                    json.loads(winner["params"]) if winner["params"] else {},
                )
                print("saved", winner["family"], "->", BUNDLE_PATH)
            sio.save_frame(model_frame, MODEL_MATRIX_PATH)
